In [ ]:
!pip install micom cobra pandas numpy optlang symengine -q
!apt-get install -y glpk-utils libglpk-dev -q
!pip install swiglpk -q
print('Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.3/50.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.8 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libamd2 libbtf1 libcamd2 libccolamd2 libcholmod3 libcolamd2 libcxsparse3
  libglpk40 libgmp-dev libgmpxx4ldbl libgraphblas-dev libgraphblas6 libklu1
  libldl2 libmetis5 libmongoose2 librbio2 libsliplu1 libspqr2
  libsuitesparse-dev libsuitesparseconfig5 libumfpack5
Suggested packages:
  libiodbc2-dev gmp-doc libgmp

In [ ]:
import cobra
cobra.Configuration().solver = 'glpk'
print('Solver:', cobra.Configuration().solver)

Solver: <module 'optlang.glpk_interface' from '/usr/local/lib/python3.12/dist-packages/optlang/glpk_interface.py'>


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

PROJECT   = '/content/drive/MyDrive/CECE_data_external'
AGORA_DIR = f'{PROJECT}/AGORA103'
SAVE_PATH = '/content/drive/MyDrive/CECE_results'
os.makedirs(SAVE_PATH, exist_ok=True)

print('File check:')
print(f"  AGORA folder:      {'YES' if os.path.exists(AGORA_DIR) else 'MISSING'}")
print(f"  podlesny_wide.pkl: {'YES' if os.path.exists(f'{PROJECT}/podlesny_wide.pkl') else 'MISSING'}")
print(f"  podlesny_meta:     {'YES' if os.path.exists(f'{PROJECT}/podlesny_meta_split.csv') else 'MISSING'}")

n_models = len([f for f in os.listdir(AGORA_DIR) if f.endswith('.xml')])
print(f'  AGORA models:      {n_models}')

Mounted at /content/drive
File check:
  AGORA folder:      YES
  podlesny_wide.pkl: YES
  podlesny_meta:     YES
  AGORA models:      818


In [ ]:
import shutil
import os

if os.path.exists("/content/AGORA103"):
    shutil.rmtree("/content/AGORA103")

print("Broken local AGORA removed")

!cp -r "/content/drive/MyDrive/CECE_data_external/AGORA103" /content/

print("Full AGORA copied")

Broken local AGORA removed
Full AGORA copied


In [ ]:
import os

DRIVE_AGORA = "/content/drive/MyDrive/CECE_data_external/AGORA103"

LOCAL_AGORA = "/content/AGORA103"

if not os.path.exists(LOCAL_AGORA):

    print("Copying AGORA models to local disk")

    os.system(
        f'cp -r "{DRIVE_AGORA}" /content/'
    )

    print("Done")

else:
    print("AGORA already on local disk")

AGORA_DIR = LOCAL_AGORA

n = len([
    f for f in os.listdir(AGORA_DIR)
    if f.endswith(".xml")
])

print(f"Models available locally: {n}")

AGORA already on local disk
Models available locally: 818


In [ ]:
import os

print("Drive AGORA count:")

drive_models = [
    f for f in os.listdir(
        "/content/drive/MyDrive/CECE_data_external/AGORA103"
    )
    if f.endswith(".xml")
]

print(len(drive_models))

print("\nLocal AGORA count:")

local_models = [
    f for f in os.listdir("/content/AGORA103")
    if f.endswith(".xml")
]

print(len(local_models))

Drive AGORA count:
818

Local AGORA count:
818


In [ ]:
import pandas as pd
import numpy as np

pod_wide = pd.read_pickle(f'{PROJECT}/podlesny_wide.pkl')
pod_meta = pd.read_csv(f'{PROJECT}/podlesny_meta_split.csv')

print(f'Abundance matrix: {pod_wide.shape}  (samples x species)')
print(f'Metadata:         {pod_meta.shape}')

donor_meta = pod_meta[pod_meta['Sample_Type'] == 'Donor'].copy()
donor_meta = donor_meta.drop_duplicates(subset='Donor.Name')
donor_ids  = [d for d in donor_meta['Name'].values if d in pod_wide.index]

donor_ids = donor_ids[:30]

print(f'\nUnique donors to simulate: {len(donor_ids)}')
print(f'Example: {donor_ids[:3]}')

Abundance matrix: (1419, 860)  (samples x species)
Metadata:         (1416, 21)

Unique donors to simulate: 30
Example: ['FRICKE_16B', 'FAT_DON_11_22_0_N', 'CHANDONOR']


In [ ]:
from micom import Community
import warnings
warnings.filterwarnings('ignore')

def build_agora_lookup(folder):
    lookup = {}
    for f in os.listdir(folder):
        if not f.endswith('.xml'):
            continue
        parts = f.replace('.xml', '').split('_')
        if len(parts) >= 2:
            lookup[parts[0] + '_' + parts[1]] = os.path.join(folder, f)
    return lookup


def smart_match(species, lookup):
    if species in lookup:
        return lookup[species]
    genus = species.split('_')[0]
    for key in lookup:
        if key.startswith(genus + '_'):
            return lookup[key]
    return None


def build_community(profile, sample_id, lookup, min_abund=0.01):
    profile    = profile[profile > min_abund]
    rows       = []
    used_files = set()

    for species, abund in profile.items():
        mf = smart_match(species, lookup)
        if mf is None or mf in used_files:
            continue
        used_files.add(mf)
        rows.append({
            'id':        species,
            'genus':     species.split('_')[0],
            'species':   species,
            'abundance': float(abund),
            'file':      mf
        })

    if len(rows) < 3:
        return None

    tax              = pd.DataFrame(rows)
    tax['abundance'] = tax['abundance'] / tax['abundance'].sum()

    try:
        return Community(tax, solver='glpk', progress=False)
    except Exception as e:
        print(f'  {sample_id}: build failed -> {e}')
        return None


def build_medium(com, uptake=10.0):
    medium = {}
    for rxn in com.exchanges:
        if rxn.lower_bound < 0:
            medium[rxn.id] = uptake
    boost = ['glc', 'fru', 'lcts', 'nh4', 'pi', 'h2o',
             'so4', 'k_', 'na1', 'ca2', 'mg2', 'fe2', 'cl_']
    for rid in medium:
        if any(k in rid.lower() for k in boost):
            medium[rid] = 100.0
    for rid in list(medium):
        if 'o2' in rid.lower():
            medium[rid] = 0.0
    return medium


agora_lookup = build_agora_lookup(AGORA_DIR)
print(f'AGORA indexed: {len(agora_lookup)}')

AGORA indexed: 668


In [ ]:
GROWTH_FILE   = f'{SAVE_PATH}/micom_growth_results.csv'
EXCHANGE_FILE = f'{SAVE_PATH}/micom_exchange_results.csv'

if os.path.exists(GROWTH_FILE):
    done      = pd.read_csv(GROWTH_FILE)['sample_id'].unique()
    remaining = [d for d in donor_ids if d not in done]
    print(f'Already done: {len(done)}  |  Remaining: {len(remaining)}')
else:
    remaining = list(donor_ids)
    print(f'Starting fresh — {len(remaining)} donors')

Already done: 30  |  Remaining: 0


In [ ]:
import gc

test_sid = remaining[0]
print(f'Sanity check: {test_sid}')

profile = pod_wide.loc[test_sid]
if isinstance(profile, pd.DataFrame):
    profile = profile.iloc[0]

com = build_community(profile, test_sid, agora_lookup, min_abund=0.01)
com.medium = build_medium(com)

sol = com.optimize(fluxes=True, pfba=True)

print(f'Status: {sol.status}')

if sol.members is not None:
    print(f'Species in sol.members: {len(sol.members)}')

ex_data = []
for rxn in com.reactions:
    if 'EX_' not in rxn.id:
        continue
    try:
        flux = rxn.forward_variable.primal - rxn.reverse_variable.primal
        if abs(flux) > 1e-8:
            ex_data.append({'metabolite': rxn.id, 'flux': flux})
    except Exception:
        continue

print(f'Exchange fluxes found: {len(ex_data)}')
print(f'\nTop 5 secretions (flux > 0):')
sec = [(r['metabolite'], r['flux']) for r in ex_data if r['flux'] > 0]
sec.sort(key=lambda x: x[1], reverse=True)
for m, f in sec[:5]:
    print(f'  {m}: {f:.4f}')

del com
gc.collect()

if len(ex_data) > 100:
    print('\nSANITY CHECK PASSED — proceed to Cell 9')
else:
    print('\nWARNING: low exchange count — check your AGORA models')

IndexError: list index out of range

In [ ]:
import gc, warnings, os
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

BATCH_SIZE = 5

batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]
print(f'{len(batches)} batches x {BATCH_SIZE} donors\n')

total_growth = total_exchange = total_failed = 0

for bn, batch in enumerate(batches):
    print(f'Batch {bn+1}/{len(batches)}')

    bg, be = [], []

    for sid in batch:
        if sid not in pod_wide.index:
            continue

        profile = pod_wide.loc[sid]
        if isinstance(profile, pd.DataFrame):
            profile = profile.iloc[0]

        com = build_community(profile, sid, agora_lookup, min_abund=0.01)
        if com is None:
            print(f'  {sid}: too few species — skip')
            total_failed += 1
            continue

        try:
            com.medium = build_medium(com)

            sol = com.optimize(fluxes=True, pfba=True)

            if sol is None or sol.status != 'optimal':
                print(f'  {sid}: {getattr(sol, "status", "None")} — skip')
                total_failed += 1
                continue

            if sol.members is not None:
                for sp, row in sol.members.iterrows():
                    if sp == 'medium':
                        continue
                    for col in ['growth_rate', 'Growth rate', 'growth']:
                        if col in row.index:
                            gr = row[col]
                            if gr is not None and not np.isnan(float(gr)):
                                bg.append({
                                    'sample_id':   sid,
                                    'species':     sp,
                                    'growth_rate': round(float(gr), 6)
                                })
                            break

            for rxn in com.reactions:
                if 'EX_' not in rxn.id:
                    continue
                try:
                    flux = (
                        rxn.forward_variable.primal
                        - rxn.reverse_variable.primal
                    )
                    if abs(flux) > 1e-8:
                        parts   = rxn.id.split('__')
                        met_id  = parts[0]
                        species = parts[1] if len(parts) > 1 else 'community'

                        be.append({
                            'sample_id':  sid,
                            'species':    species,
                            'metabolite': met_id,
                            'flux':       round(float(flux), 8)
                        })
                except Exception:
                    continue

            n_grown  = sum(1 for r in bg if r['sample_id'] == sid)
            n_fluxes = sum(1 for r in be if r['sample_id'] == sid)
            print(f'  {sid}: OK — {n_grown} species, {n_fluxes} exchange fluxes')

        except Exception as e:
            print(f'  {sid}: error -> {e}')
            total_failed += 1

        finally:
            del com
            gc.collect()

    if bg:
        pd.DataFrame(bg).to_csv(
            GROWTH_FILE, mode='a',
            header=not os.path.exists(GROWTH_FILE),
            index=False
        )
        total_growth += len(bg)

    if be:
        pd.DataFrame(be).to_csv(
            EXCHANGE_FILE, mode='a',
            header=not os.path.exists(EXCHANGE_FILE),
            index=False
        )
        total_exchange += len(be)

    print(f'  Saved — growth: {total_growth}  exchanges: {total_exchange}  failed: {total_failed}\n')

print('All batches complete')

In [ ]:
gdf = pd.read_csv(GROWTH_FILE)
edf = pd.read_csv(EXCHANGE_FILE)

print('Growth results')
print(f'Records:  {len(gdf)}')
print(f'Donors:   {gdf["sample_id"].nunique()}')
print(f'Species:  {gdf["species"].nunique()}')
print(f'\nGrowth rate stats:')
print(gdf['growth_rate'].describe().round(4))

print(f'\n Exchange results')
print(f'Records:     {len(edf)}')
print(f'Donors:      {edf["sample_id"].nunique()}')
print(f'Species:     {edf["species"].nunique()}')
print(f'Metabolites: {edf["metabolite"].nunique()}')
print(f'Secretions (flux > 0): {(edf["flux"] > 0).sum()}')
print(f'Consumptions (flux < 0): {(edf["flux"] < 0).sum()}')

print(f'\nTop secreting species:')
print(
    edf[edf['flux'] > 0]
    .groupby('species')['flux']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

In [ ]:
from google.colab import files

print('Downloading')
files.download(GROWTH_FILE)
files.download(EXCHANGE_FILE)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os

SAVE_PATH     = '/content/drive/MyDrive/CECE_results'
EXCHANGE_FILE = f'{SAVE_PATH}/micom_exchange_results.csv'
SCFA_FILE     = f'{SAVE_PATH}/micom_scfa_scores.csv'

if not os.path.exists(EXCHANGE_FILE):
    print(f"ERROR: {EXCHANGE_FILE} not found")
    print("Check your SAVE_PATH above")
else:
    edf = pd.read_csv(EXCHANGE_FILE)
    print(f"Exchange records loaded: {len(edf)}")
    print(f"Columns: {edf.columns.tolist()}")
    print(f"Sample metabolite IDs: {edf['metabolite'].head(5).tolist()}")


    SCFA_PATTERNS = {
    'butyrate':   ['ex_but(e)', 'ex_but_e', 'ex_but__l_e'],
    'propionate': ['ex_ppa(e)', 'ex_ppa_e', 'ex_prop(e)', 'ex_prop_e'],
    'acetate':    ['ex_ac(e)',  'ex_ac_e'],
    'lactate':    ['ex_lac_L(e)', 'ex_lac_D(e)', 'ex_lac_'],
    'succinate':  ['ex_succ(e)', 'ex_succ_'],
    'formate':    ['ex_for(e)',  'ex_for_'],
    }

    rows = []
    print("\nSearching for SCFA reactions...")

    for scfa_name, pat in SCFA_PATTERNS.items():
        matches    = edf[edf['metabolite'].str.contains(pat, na=False)]
        secretions = matches[matches['flux'] > 0]
        n_found    = len(secretions)
        print(f"  {scfa_name:12s} ({pat}): {len(matches)} total, "
              f"{n_found} secretions")

        if n_found > 0:
            per_sp = (
                secretions
                .groupby('species')['flux']
                .mean()
                .reset_index()
            )
            per_sp.columns = ['species', 'mean_flux']
            per_sp['scfa'] = scfa_name
            rows.append(per_sp)

    if rows:
        scfa_long = pd.concat(rows, ignore_index=True)
        scfa_wide = scfa_long.pivot_table(
            index='species',
            columns='scfa',
            values='mean_flux',
            aggfunc='mean',
            fill_value=0
        ).reset_index()

        scfa_wide.to_csv(SCFA_FILE, index=False)

        print(f"\nSCFA scores saved to Drive: {scfa_wide.shape}")
        print(f"SCFAs captured: {[c for c in scfa_wide.columns if c != 'species']}")
        print(f"\nTop 5 butyrate producers:")
        if 'butyrate' in scfa_wide.columns:
            print(scfa_wide[['species','butyrate']]
                  .sort_values('butyrate', ascending=False)
                  .head(5).to_string(index=False))

        from google.colab import files
        files.download(SCFA_FILE)
        print(f"\nDownloading micom_scfa_scores.csv...")
        print("Put it in: data/processed/")

    else:
        print("\nNo SCFA patterns matched in your exchange data")
        print("Check metabolite column values above")
        print("\nAll unique metabolite prefixes (first 6 chars):")
        prefixes = edf['metabolite'].str[:6].value_counts().head(20)
        print(prefixes.to_string())

Mounted at /content/drive
Exchange records loaded: 39434
Columns: ['sample_id', 'species', 'metabolite', 'flux']
Sample metabolite IDs: ['EX_12dgr180(e)', 'EX_12dhchol(e)', 'EX_26dap_M(e)', 'EX_4abut(e)', 'EX_C02528(e)']

Searching for SCFA reactions...


TypeError: unhashable type: 'list'